**Initial Set-Up**

1. Install Anaconda3 using GUI installer/install Miniforge

2. Run installation commands to install Mamba

    conda config --add channels conda-forge

    conda update -n base --all

    conda install -n base mamba


3. Use Mamba to install PyImageJ
    mamba install -c conda-forge pyimagej openjdk=11

**OR, use Pip**

    Install Python 3
    
    Install OpenJDK 8 or OpenJDK 11

    Install Maven

    Run: pip install pyimagej


**Errors Encountered Previously and Currently**

Install Java
   IF: You recieve an error indicating the program's inability to resolve Java, 
   then install the official Java Development Kit (JDK from the website) 
   https://www.oracle.com/java/technologies/downloads/?er=221886#jdk25. 

Original ImageJ not available    
 - Do not change initialization line. 

Re-running cell with initialization code (see below) leads to problems with initialization.
 - Guarding implemented ("if 'ij' not in globals()...")

 **Todo** 
  - Take measurements from an image and append to a .csv file --> complete
  - Implement looping through a folder. --> complete
  - Dump csv --> .pkl file --> complete
  - Implement particle analysis 

In [21]:
# ! mamba remove -n base pyimagej
# ! pip install pyimagej

from IPython.display import Image, display
from pathlib import Path
import imagej
import scyjava
from scyjava import jimport
import pandas as pd
import numpy as np
import os

print("Dependencies present")

# Set Random Image for Testing
test_image_pth = "content/Images1/noise 3 21001.0.png"

# Pre-sets
path_in1 = "./content/Images1/"
path_in2 = "./content/Images2/"
path_in3 = "./content/Images3/"

path_out_csv = "./data/feats_extracted.csv"

# Set Java heap size = 6 gb.
# scyjava.config.add_option('-Xmx6g')

# Folder containing PNG images
# Select one of the input paths (path_in1, path_in2, path_in3)
input_dir = path_in2
output_csv = Path(path_out_csv)

print("I/O Set.")

Dependencies present
I/O Set.


In [22]:
# Initialize ImageJ
# Java Virtual Machine Guarding
if 'ij' not in globals():
    ij = imagej.init('sc.fiji:fiji', headless=False, add_legacy=True)
else:
    print("JVM instance already running.")

print(f"Ensure ImageJ Legacy layer is available. Status: {ij.legacy.isActive()}")

## Clear 
ij.IJ.run("Clear Results")
Summary = ij.ResultsTable.getResultsTable("Summary")
if Summary is not None:
    Summary.reset()

## GET PARTICLE DATA

print("Particle Start...")

print(f"Analyzing measurements on all images from folder {input_dir}")

Prefs = scyjava.jimport('ij.Prefs'); 
Prefs.blackBackground = False


counter = 0
for file in os.listdir(input_dir):
    counter += 1

    if counter % 10 == 0:
        print(counter)

    # Can be deleted for production purposes later. Implemented for efficiency.
    if counter > 100:
        break

    full_path = str(input_dir) + '/' + file

    # Load test image
    dataset = ij.IJ.openImage(full_path)

    # Select entire image
    ij.IJ.run(dataset, "Select All", "")

    ij.IJ.run(dataset, "8-bit", "")
    ij.IJ.setAutoThreshold(dataset, "Default")
    ij.IJ.run(dataset, "Convert to Mask", "")

    ij.IJ.run(
        dataset,
        "Analyze Particles...",
        "display clear summarize overlay record"
    )

print("Particles analyzed.")

ResultsTable = ij.ResultsTable.getResultsTable("Summary")

print("Get Summary Table")

# Returns tab delimited. .split ensure grouping
cols = list(ResultsTable.getColumnHeadings().split('\t'))
print(cols)

all_data = {}

for col in cols:
    col = str(col)
    if col == 'Slice':
        col_data = [ResultsTable.getStringValue(col, i) for i in range(ResultsTable.size())]
        for i in range(len(col_data)):
            col_data[i] = str(col_data[i])
        all_data[col] = col_data
    else:
        col_data = ResultsTable.getColumn(col)
        all_data[col] = col_data

particle_summary_df__whitebg = pd.DataFrame(data=all_data)
particle_summary_df__whitebg.rename(columns={'Slice' : 'Path'}, inplace=True)

particle_summary_df__whitebg

JVM instance already running.
Ensure ImageJ Legacy layer is available. Status: True
Particle Start...
Analyzing measurements on all images from folder ./content/Images2/
10
20
30
40
50
60
70
80
90
100
Particles analyzed.
Get Summary Table
['Slice', 'Count', 'Total Area', 'Average Size', '%Area', 'Mean', 'Mode', 'Perim.', 'Major', 'Minor', 'Angle', 'Circ.', 'Solidity', 'Feret', 'FeretX', 'FeretY', 'FeretAngle', 'MinFeret', 'IntDen', 'Median', 'Skew', 'Kurt']


,Path,Count,Total Area,Average Size,%Area,Mean,Mode,Perim.,Major,Minor,...,Solidity,Feret,FeretX,FeretY,FeretAngle,MinFeret,IntDen,Median,Skew,Kurt
0,noise 3 20401.0.png,1.0,40000.0,40000.000000,100.0000,255.0,255.0,797.656860,225.675827,225.675827,...,1.000000,282.842712,0.000000,0.000000,135.00000,200.000000,1.020000e+07,255.0,NaN,NaN
1,noise 3 20402.0.png,1.0,37062.0,37062.000000,92.6550,255.0,255.0,883.823364,217.470444,216.989502,...,0.926550,282.842712,0.000000,0.000000,135.00000,200.000000,9.450810e+06,255.0,NaN,NaN
2,noise 3 20403.0.png,0.0,0.0,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,noise 3 20404.0.png,0.0,0.0,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,noise 3 20405.0.png,26.0,543.0,20.884615,1.3575,255.0,255.0,15.420893,5.495724,4.817807,...,0.924214,6.056007,106.269231,100.692308,117.81557,4.615385,5.325577e+03,255.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,noise 3 20497.0.png,1.0,40000.0,40000.000000,100.0000,255.0,255.0,797.656860,225.675827,225.675827,...,1.000000,282.842712,0.000000,0.000000,135.00000,200.000000,1.020000e+07,255.0,NaN,NaN
96,noise 3 20498.0.png,1.0,40000.0,40000.000000,100.0000,255.0,255.0,797.656860,225.675827,225.675827,...,1.000000,282.842712,0.000000,0.000000,135.00000,200.000000,1.020000e+07,255.0,NaN,NaN
97,noise 3 20499.0.png,0.0,0.0,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,noise 3 20500.0.png,1.0,40000.0,40000.000000,100.0000,255.0,255.0,797.656860,225.675827,225.675827,...,1.000000,282.842712,0.000000,0.000000,135.00000,200.000000,1.020000e+07,255.0,NaN,NaN


In [20]:
particle_summary_df__blackbg.to_csv("./data/particle_summary_df__blackbg.csv")
particle_summary_df__whitebg.to_csv("./data/particle_summary_df__whitebg.csv")

In [19]:
# Initialize ImageJ
# Java Virtual Machine Guarding
if 'ij' not in globals():
    ij = imagej.init('sc.fiji:fiji', headless=False, add_legacy=True)
else:
    print("JVM instance already running.")

print(f"Ensure ImageJ Legacy layer is available. Status: {ij.legacy.isActive()}")

## Clear 
ij.IJ.run("Clear Results")
Summary = ij.ResultsTable.getResultsTable("Summary")
if Summary is not None:
    Summary.reset()

## GET PARTICLE DATA

print("Particle Start...")

print(f"Analyzing measurements on all images from folder {input_dir}")

Prefs = scyjava.jimport('ij.Prefs'); 
Prefs.blackBackground = True

counter = 0
for file in os.listdir(input_dir):
    counter += 1

    if counter % 10 == 0:
        print(counter)

    # Can be deleted for production purposes later. Implemented for efficiency.
    if counter > 100:
        break

    full_path = str(input_dir) + '/' + file

    # Load test image
    dataset = ij.IJ.openImage(full_path)

    # Select entire image
    ij.IJ.run(dataset, "Select All", "")

    ij.IJ.run(dataset, "8-bit", "")
    ij.IJ.setAutoThreshold(dataset, "Default")
    ij.IJ.run(dataset, "Convert to Mask", "")

    ij.IJ.run(
        dataset,
        "Analyze Particles...",
        "display clear summarize overlay record"
    )

print("Particles analyzed.")

ResultsTable = ij.ResultsTable.getResultsTable("Summary")

print("Get Summary Table")

# Returns tab delimited. .split ensure grouping
cols = list(ResultsTable.getColumnHeadings().split('\t'))
print(cols)

all_data = {}

for col in cols:
    col = str(col)
    if col == 'Slice':
        col_data = [ResultsTable.getStringValue(col, i) for i in range(ResultsTable.size())]
        for i in range(len(col_data)):
            col_data[i] = str(col_data[i])
        all_data[col] = col_data
    else:
        col_data = ResultsTable.getColumn(col)
        all_data[col] = col_data

particle_summary_df__blackbg = pd.DataFrame(data=all_data)
particle_summary_df__blackbg.rename(columns={'Slice' : 'Path'}, inplace=True)

particle_summary_df__blackbg

JVM instance already running.
Ensure ImageJ Legacy layer is available. Status: True
Particle Start...
Analyzing measurements on all images from folder ./content/Images1/
10
20
30
40
50
60
70
80
90
100
Particles analyzed.
Get Summary Table
['Slice', 'Count', 'Total Area', 'Average Size', '%Area', 'Mean', 'Mode', 'Perim.', 'Major', 'Minor', 'Angle', 'Circ.', 'Solidity', 'Feret', 'FeretX', 'FeretY', 'FeretAngle', 'MinFeret', 'IntDen', 'Median', 'Skew', 'Kurt']


,Path,Count,Total Area,Average Size,%Area,Mean,Mode,Perim.,Major,Minor,...,Solidity,Feret,FeretX,FeretY,FeretAngle,MinFeret,IntDen,Median,Skew,Kurt
0,noise 3 21001.0.png,1.0,39599.0,39599.000000,98.9975,255.0,255.0,815.053833,224.616592,224.466995,...,0.989975,282.842712,0.000000,0.000000,135.000000,200.000000,1.009774e+07,255.0,NaN,NaN
1,noise 3 21002.0.png,1.0,40000.0,40000.000000,100.0000,255.0,255.0,797.656860,225.675827,225.675827,...,1.000000,282.842712,0.000000,0.000000,135.000000,200.000000,1.020000e+07,255.0,NaN,NaN
2,noise 3 21003.0.png,0.0,0.0,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,noise 3 21004.0.png,1.0,40000.0,40000.000000,100.0000,255.0,255.0,797.656860,225.675827,225.675827,...,1.000000,282.842712,0.000000,0.000000,135.000000,200.000000,1.020000e+07,255.0,NaN,NaN
4,noise 3 21005.0.png,1.0,38443.0,38443.000000,96.1075,255.0,255.0,830.793945,221.328400,221.151672,...,0.961075,282.842712,0.000000,0.000000,135.000000,200.000000,9.802965e+06,255.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,noise 3 21096.0.png,0.0,0.0,NaN,0.0000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,noise 3 21097.0.png,11.0,790.0,71.818182,1.9750,255.0,255.0,51.175089,18.514958,4.643602,...,0.677884,21.676073,82.454545,96.727273,108.450164,6.831765,1.831364e+04,255.0,NaN,NaN
97,noise 3 21098.0.png,1.0,35821.0,35821.000000,89.5525,255.0,255.0,880.575684,213.683243,213.440750,...,0.895525,282.842712,0.000000,0.000000,135.000000,200.000000,9.134355e+06,255.0,NaN,NaN
98,noise 3 21099.0.png,1.0,38192.0,38192.000000,95.4800,255.0,255.0,840.735046,220.546478,220.486694,...,0.954800,282.842712,0.000000,0.000000,135.000000,200.000000,9.738960e+06,255.0,NaN,NaN


In [ ]:
# Initialize ImageJ
# Java Virtual Machine Guarding
if 'ij' not in globals():
    ij = imagej.init('sc.fiji:fiji', headless=False, add_legacy=True)
else:
    print("JVM instance already running.")

print(f"Ensure ImageJ Legacy layer is available. Status: {ij.legacy.isActive()}")


In [ ]:
print(f"Analyzing measurements on all images from folder {input_dir}")

counter = 0
for file in os.listdir(input_dir):
    counter += 1
    
    if counter % 1000 == 0:
        print(counter)

    # Can be deleted for production purposes later. Implemented for efficiency.
    if counter > 2000:
        break

    full_path = str(input_dir) + '/' + file

    # Load test image
    dataset = ij.IJ.openImage(full_path)

    # Display the image within Jupyter Lab for intuitive purposes 
    #display(Image(full_path))

    # Select entire image
    ij.IJ.run(dataset, "Select All", "")

    # Set Measurements
    ij.IJ.run("Set Measurements...", """area mean standard modal min centroid center perimeter bounding fit shape feret's integrated median skewness kurtosis area_fraction stack display add redirect=None decimal=3""")

    # Extract Measurements from the image
    ij.IJ.run(dataset, "Measure", "")   

print("Measurments complete.") 
print("Do not close ImageJ Results window before compiling the rest of the program.")
print("Current limitation: do not run this block with all three folders because dynamic folder column addition is not yet implemented. Run it with one folder, save to dataframe, then rerun.")

In [ ]:
# Get Results 
ResultsTable = ij.ResultsTable.getResultsTable()

# Returns tab delimited. .split ensure grouping
cols = list(ResultsTable.getColumnHeadings().split('\t'))
cols.remove(' ')

all_data = {}

for col in cols:
    col = str(col)
    if col == 'Label':
        col_data = [ResultsTable.getStringValue(col, i) for i in range(ResultsTable.size())]
        for i in range(len(col_data)):
            col_data[i] = str(col_data[i])
        all_data[col] = col_data
    else:
        col_data = ResultsTable.getColumn(col)
        all_data[col] = col_data

feats_extracted_df = pd.DataFrame(data=all_data)
feats_extracted_df.rename(columns={'Label' : 'Path'}, inplace=True)
feats_extracted_df['Folder'] = f'{input_dir}'


feats_extracted_df

In [ ]:
feats_extracted_df.to_csv(path_out_csv)